# Chapter 7. Microsoft's GraphRAG Implementation

A key innovation of Microsoft's GraphRAG is its use of an LLM to build a knowledge graph through a two-stage process.
- First, entities and relationships are extracted and summarized from source documents to form the foundation of the knowledge graph.
- Then, once the knowledge graph has been constructed, graph communities are detected, and domain-specific summaries are generated for groups of closely related entities.

This layered approach transforms fragmented pieces of information from various text chunks into a cohesive and organized representation of information about specified entities, relationships, and communities.

The figure below illustrates the architecture of Microsoft's GraphRAG implementation.

![MS GraphRAG](./imgs/ms-graphrag.png)

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import json
import requests
from tqdm import tqdm
from typing import List, Dict

from utils.utils import neo4j_driver, num_tokens_from_string, chunk_text, chat, embed
import ch07_tools


## Dataset Selection

We will use "The Odyssey" to evaluate MS GraphRAG.

In [3]:
url = "https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
response = requests.get(url)

## Graph Indexing

### Chunking

In [4]:
def chunk_into_books(text: str) -> List[str]:
    # Remove prefaces and footnotes, then split into books
    return (
        text.split("PREFACE TO FIRST EDITION")[2]
            .split("FOOTNOTES")[0]
            .strip()
            .split("\nBOOK")[1:]
    )


In [5]:
books = chunk_into_books(response.text)

In [6]:
# Check number of tokens in each book
token_counts = [num_tokens_from_string(book) for book in books]

print(
    f"""There are {len(token_counts)} books with token sizes:
- avg {sum(token_counts) / len(token_counts):.2f}
- min {min(token_counts)}
- max {max(token_counts)}"""
)

There are 24 books with token sizes:
- avg 6466.62
- min 4421
- max 10701


In [7]:
# Assume we want chunks of around 1000 tokens with an overlap of 40 tokens
chunked_books = [chunk_text(book, 1000, 40) for book in books]

### Entity and Relationship Extraction

We will borrow the MS GraphRAG prompts for entity and relationship extraction. This is the `GRAPH_EXTRACTION_PROMPT` in `ch07_tools.py`.

We need to pass the list of entity types `entity_types` to the prompt. We will use the following entity types:

In [8]:
ENTITY_TYPES = [
    "PERSON",
    "ORGANIZATION",
    "LOCATION",
    "GOD",
    "EVENT",
    "CREATURE",
    "WEAPON_OR_TOOL"
]

In [9]:
def extract_entities(text: str) -> List[Dict]:
    # Construct the prompt
    messages = [
        {
            "role": "user",
            "content": ch07_tools.create_extraction_prompt(ENTITY_TYPES, text)
        }
    ]

    # Call the chat API
    output = chat(messages, model='gpt-5.1')
    # Construct JSON from the output
    return ch07_tools.parse_extraction_output(output)

In [10]:
# Extract entities from all chunks
number_of_books = 1

for book_i, book in enumerate(
    # Define the number of books to process here
    tqdm(chunked_books[:number_of_books], desc="Processing Books")
):
    for chunk_i, chunk in enumerate(tqdm(book, desc=f"Book {book_i}", leave=False)):
        # Extract entities and relationships from the chunk
        nodes, relationship = extract_entities(chunk)

        # Import entities
        neo4j_driver.execute_query(
            ch07_tools.import_nodes_query,
            data=nodes,
            book_id=book_i,
            text=chunk,
            chunk_id=chunk_i
        )

        # Import relationships
        neo4j_driver.execute_query(
            ch07_tools.import_relationships_query,
            data=relationship,
        )

Processing Books: 100%|██████████| 1/1 [06:45<00:00, 405.59s/it]


The sample of the imported graph is shown below:

![Graph Sample](./imgs/book-0-graph.png)

After importing extracted entities and relationships into Neo4j, we can run the following query to review the imported data:

In [11]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (:`__Entity__`)
    RETURN 'entity' AS type, count(*) AS count
    UNION
    MATCH ()-[:RELATIONSHIP]->()
    RETURN 'relationship' AS type, count(*) AS count
    """
)
print([el.data() for el in data])

[{'type': 'entity', 'count': 154}, {'type': 'relationship', 'count': 311}]


MS GraphRAG focuses on extracting detailed descriptions of both entities and their relationships. We can examine the extracted descriptions for the character `ORESTES`.

In [12]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.description AS description"""
)
print([el.data()['description'] for el in data])

[['Orestes is the son of Agamemnon who killed Aegisthus in retribution for the murder of his father', 'Orestes is a man prophesied to grow up, return home, and take revenge on Aegisthus for his crimes.', 'Orestes is a man praised in song for killing Aegisthus, the murderer of his father.']]


While some descriptions repeat the same facts, they collectively contain all the key details and ensure no important information is lost across different text chunks for a specific entity.

In addition, a single pair of entities can have multiple relationships. We can explore the entity pair with the highest number of relationships:

In [13]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[:RELATIONSHIP]-(m:__Entity__)
WITH n,m, count(*) AS countOfRels
ORDER BY countOfRels DESC LIMIT 1
MATCH (n)-[r:RELATIONSHIP]-(m)
RETURN n.name AS source, m.name AS target, countOfRels, collect(r.description) AS descriptions
"""
)
print([el.data() for el in data])

[{'source': 'TELEMACHUS', 'target': 'MINERVA', 'countOfRels': 8, 'descriptions': ['Telemachus speaks quietly and intimately to Minerva during the banquet so that no man might hear', 'Telemachus converses respectfully with Minerva, treats her as a dear friend, offers her a bath and a valuable present, and is emotionally and mentally influenced by her guidance', 'Minerva plans to encourage and embolden Telemachus to call an assembly, confront the suitors, and journey for news of his father', 'Minerva is noticed first by Telemachus, who goes to the gate, takes her hand, and welcomes her as a stranger', 'Minerva speaks directly to Telemachus, reassures him about his lineage, and comments on his situation', 'Minerva has given Telemachus counsel about his intended voyage, establishing a guiding, advisory relationship between goddess and mortal', 'Minerva speaks directly to Telemachus, advising him and commenting on his need for Ulysses', 'Minerva advises Telemachus kindly as though he were h

### Entity and Relationship Summarization

To avoid inconsistencies, redundancies, and fragmentation in the extracted knowledge, MS GraphRAG merges multiple descriptions of the same entity or relationship using LLMs to generate concise summaries.

Instead of treating each description separately, the model synthesizes information from all descriptions, ensuring that key contextual details are preserved in a single, enriched representation.

We can reuse the summarization prompt from the paper, which is the `SUMMARIZE_PROMPT` in `ch07_tools.py`. Then we can generate summaries for all entities that have more than a single description.

In [14]:
# Get all entities that have more than a single description to summarize
candidate_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (e:__Entity__) WHERE size(e.description) > 1 
    RETURN e.name AS entity_name, e.description AS description_list"""
)

summaries = []
for candidate in tqdm(candidate_to_summarize, desc="Summarizing Entities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                entity_name=candidate['entity_name'],
                description_list=candidate['description_list']
            )
        }
    ]
    # Generate entity summary
    summary = chat(messages, model='gpt-5.1')
    summaries.append(
        {
            "entity": candidate['entity_name'],
            "summary": summary
        }
    )

ch07_tools.import_entity_summary(neo4j_driver, summaries)

Summarizing Entities: 100%|██████████| 27/27 [01:21<00:00,  3.02s/it]


After that, we can review the summarized description of `ORESTES`:

In [15]:
summary, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.summary AS summary""")
print(summary[0]['summary'])

ORESTES is the son of Agamemnon, prophesied to grow up, return home, and take revenge on Aegisthus for his crimes. After Aegisthus murdered Agamemnon, Orestes fulfilled this prophecy by killing Aegisthus in retribution for his father’s death. For this act of vengeance, Orestes is praised in song as the avenger of his father and the slayer of Aegisthus.


By merging multiple descriptions, we ensure that key details are preserved while reducing redundancy.

Next we will apply the same summarization process to relationships, consolidating multiple descriptions of the same relationship into a single, comprehensive summary.

In [16]:
# Retrieve pairs of nodes with more than a single relationship to summarize
rels_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (s:__Entity__)-[r:RELATIONSHIP]-(t:__Entity__)
    WHERE id(s) < id(t)
    WITH s.name AS source, t.name AS target, 
           collect(r.description) AS description_list,
           count(*) AS count
    WHERE count > 1
    RETURN source, target, description_list"""
)
rel_summaries = []
for candidate in tqdm(rels_to_summarize, desc="Summarizing relationships"):
    entity_name = f"{candidate['source']} relationship to {candidate['target']}"
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                entity_name, candidate["description_list"]
            ),
        },
    ]
    # Generate relationship summary
    summary = chat(messages, model="gpt-4o")
    rel_summaries.append({"source": candidate["source"], "target": candidate["target"], "summary": summary})

# Store relationship summaries
ch07_tools.import_rels_summary(neo4j_driver, rel_summaries)

Summarizing relationships: 100%|██████████| 30/30 [01:05<00:00,  2.19s/it]


By merging relationship descriptions, the process ensures that key interactions between entities are captured comprehensively while eliminating redundancy.

We can evaluate the generated relationship between `TELEMACHUS` and `MINERVA`:

In [17]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[r:SUMMARIZED_RELATIONSHIP]-(m:__Entity__)
WHERE n.name = 'TELEMACHUS' AND m.name = 'MINERVA'
RETURN r.summary AS description
"""
)
print(data[0]["description"])

The relationship between Minerva and Telemachus is characterized by a deep and influential bond, where Minerva plays the role of a guiding and advisory figure to the young Telemachus. During a banquet, Telemachus speaks quietly and intimately with Minerva, ensuring that no other man might overhear their conversation, which underscores the trust and respect he holds for her. He treats Minerva with great respect and affection, offering her a bath and a valuable present, and regards her as a dear friend. Minerva, in turn, plans to embolden Telemachus to take decisive actions, such as calling an assembly, confronting the suitors, and embarking on a journey to seek news of his father, Ulysses.

Telemachus is the first to notice Minerva, welcoming her warmly as a stranger at the gate, which highlights his hospitable nature. Minerva reassures Telemachus about his lineage and comments on his current situation, providing him with the emotional and mental support he needs. She advises him direct

With the consolidated summaries for both entities and relationships, we have successfully completed the first stage of MS GraphRAG indexing. By merging information
across text chunks, we have created a more coherent and enriched representation of the extracted knowledge.

### Community Detection and Summarization

The second stage of the graph-indexing process focuses on **community detection** and **summarization**.

A **community** is a group of entities that are more densely connected to each other than to the rest of the graph. The figure below illustrates the concept of communities in a knowledge graph.

![Communities](./imgs/community-detection.png)

Each community shown above represents a set of densely connected entities with stronger internal relationships. Some communities are well integrated into the overall graph, while others appear more isolated, forming disconnected subgraphs. Identifying these clusters helps reveal underlying structures, themes, or key groups within the dataset.

By detecting and summarizing these communities, we can capture higher-level relationships and insights that go beyond indi vidual entity connections.

We will apply the *Louvain method*, a community detection algorithm, to identify communities in the graph. The detected communities are then stored as a node property for downstream processing.

Make sure the "Graph Data Science" plugin is installed in the Neo4j instance to run the community detection algorithm.

In [20]:
community_distribution = ch07_tools.calculate_communities(neo4j_driver)
print(f"There are {community_distribution['communityCount']} communities with distribution: {community_distribution['communityDistribution']}")

There are 9 communities with distribution: {'p1': 5, 'max': 38, 'p5': 5, 'p90': 38, 'p50': 13, 'p95': 38, 'p10': 5, 'p75': 24, 'p99': 38, 'p25': 9, 'min': 5, 'mean': 17.11111111111111, 'p999': 38}


Louvain is not deterministic, meaning that even with the same input, the detected communities may vary slightly between runs due to the algorithm’s optimization process.

Next we can apply the summarization prompt to generate concise overviews of each detected community, which is the `COMMUNITY_REPORT_PROMPT` in `ch07_tools.py`. 

This will guide the AI assistant in generating structured summaries of detected communities, ensuring they capture key entities, relationships, and notable insights. The goal is to produce high-quality summaries that can be effectively used downstream for RAG.

With the communities identified and a structured summarization prompt in place, we can now generate comprehensive summaries for each detected community. These community summaries consolidate key entities, relationships, and significant insights.

In [21]:
# Retrieve community information from database
community_info, _, _ = neo4j_driver.execute_query(ch07_tools.community_info_query)

communities = []
for community in tqdm(community_info, desc="Summarizing Communities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_community_prompt(
                nodes=community['nodes'],
                relationships=community['rels']
            )
        }
    ]

    summary = chat(messages, model='gpt-5.1')
    communities.append(
        {
            # Parse output into dictionary
            'community': json.loads(ch07_tools.extract_json(summary)),
            'communityId': community['communityId'],
            'nodes': [el['id'] for el in community['nodes']]
        }
    )
# Store community summaries in the database
neo4j_driver.execute_query(ch07_tools.import_community_query, data=communities)

Summarizing Communities: 100%|██████████| 9/9 [08:11<00:00, 54.64s/it]


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002B210F222D0>, keys=[])

We can examine an example of a generated community summary:

In [22]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (c:__Community__)
WITH c, count {(c)<-[:IN_COMMUNITY]-()} AS size
ORDER BY size DESC LIMIT 1
RETURN c.title AS title, c.summary AS summary
"""
)
print(data[0]["title"])
print(data[0]["summary"])

Minerva–Telemachus Guidance Network in Ithaca and the Achaean World
This community centers on the goddess Minerva (Athena) and the young noble Telemachus, embedded in a wider network of Olympian gods, Achaean warriors, and domestic and travel-related settings. Minerva acts as divine strategist and protector, advocating before Jupiter for Ulysses’ return, descending from Olympus to Ithaca, disguising herself as the mortal Mentes, and guiding Telemachus’ political awakening and planned voyage to Pylos and Sparta. Telemachus, in turn, responds with growing assertiveness, summoning the Achaeans, confronting the suitors’ abuses in his household, and preparing to consult Nestor in Pylos and Menelaus in Sparta for news of his father, while his domestic environment—servants, house, tower room, and security mechanisms—illustrates the social and material context of his transition to leadership. The broader network includes the Achaeans as a collective, the cities of Pylos and Sparta, and figures

This concludes the graph indexing process for MS GraphRAG.

## Graph Retrievers

The graph retriever stage focuses on retrieving relevant information from the structured graph to answer queries effectively. We will focus on two primary approaches: local search and global search.
- *Local search* retrieves information from entities closely connected within a detected community, whereas
- *Global search* considers the entire graph structure to find the most relevant information.

### Global Search

Global search in GraphRAG uses community summaries as intermediate responses to efficiently answer queries that require aggregating information across the entire dataset.

Instead of retrieving individual chunks of text based on vector similarity, this method utilizes precomputed community-level summaries to generate a structured response, as shown in the figure below.

![Global Search](./imgs/global-search.png)

The entire process in the figure above follows a map-reduce approach:
- **Map step**: Given a user query and, optionally, the conversation history, GraphRAG retrieves LLM-generated community reports from a specified level in the graph’s community hierarchy. In our implementation, the graph is structured with a single level of communities, meaning all detected groups exist at the same hierarchical depth. These reports are segmented into manageable text chunks, and each chunk is processed by the LLM to produce an intermediate response.
- **Reduce step**: The most important points across all intermediate responses are filtered and aggregated.  These refined insights then serve as the final context
for the LLM, which synthesizes a cohesive answer to the user query. By structuring the dataset into semantically meaningful clusters, GraphRAG enables efficient and cohesive retrieval, even for broad, thematic queries.

The map step uses the system prompt `MAP_SYSTEM_PROMPT` in `ch07_tools.py`. The map system prompt instructs the LLM to extract key points from the provided context in response to a user query. Each key point includes a description and a importance score (0-100) reflecting its relevance to the query.

The reduce step uses the system prompt `REDUCE_SYSTEM_PROMPT` in `ch07_tools.py`. The reduce system prompt directs the LLM to synthesize key points from multiple analyst reports, which are ranked by importance. The response must be formatted in Markdown, be structured appropriately for the target length and format, and exclude irrelevant details. It preserves all referenced data while avoiding speculative answers. The final output integrates and refines insights from the reports into a coherent, comprehensive response to the user query.

Now we can combine the map and reduce steps into a global search function:

In [24]:
def global_retriever(query: str, rating_threshold: float = 5) -> str:
    # Get all communities above the rating threshold
    community_data, _, _ = neo4j_driver.execute_query(
        """
    MATCH (c:__Community__)
    WHERE c.rating >= $rating
    RETURN c.summary AS summary
    """,
        rating=rating_threshold,
    )
    print(f"Got {len(community_data)} community summaries")

    intermediate_results = []
    for community in tqdm(community_data, desc="Processing Communities"):
        # For each community, get an intermediate response
        intermediate_messages = [
            {
                'role': 'system',
                'content': ch07_tools.get_map_system_prompt(community['summary'])
            },
            {
                'role': 'user',
                'content': query
            }
        ]
        intermediate_response = chat(intermediate_messages, model='gpt-5.1')
        intermediate_results.append(intermediate_response)

    # Generate a final answer using all the intermediate responses as context
    final_messages = [
        {
            'role': 'system',
            'content': ch07_tools.get_reduce_system_prompt(intermediate_results)
        },
        {
            'role': 'user',
            'content': query
        }
    ]

    summary = chat(final_messages, model='gpt-5.1')
    return summary

The `global_retriever` function follows three processes:
- *Retrieve relevant communities*
- *Generate intermediate responses*
- *Aggregate and generate final answer*

In [25]:
# Test global retriever
print(global_retriever("What is this story about?"))

Got 7 community summaries


Processing Communities: 100%|██████████| 7/7 [00:37<00:00,  5.38s/it]


## Overview of the Story

The story is an early portion of the *Odyssey*–tradition: it is about the mortal hero Ulysses (Odysseus) and the long‑delayed, fiercely contested return to his home in Ithaca after the Trojan War. Rather than simply listing his travels, it focuses on how his fate is shaped, delayed, and debated by gods and mortals alike, and how his absence throws his household and kingdom into crisis [Data: Reports (1)].

Around this central thread, the narrative weaves together several interconnected strands: the quarrels and councils of the gods, the political breakdown in Ithaca, the coming‑of‑age of his son Telemachus, the loyalty and suffering of his wife Penelope, and the wider mythic world of seafaring peoples, monsters, and divine festivals [Data: Reports (1)].

---

## Divine Powers and Ulysses’ Fate

A key theme is the struggle among divine powers over Ulysses’ destiny. Jove (Zeus) and the immortal gods in heaven ultimately shape his fate, but other deities oppose o

### Local Search

The local search method enhances LLM responses by combining structured knowledge graph data with unstructured text from source documents.

This approach is effective for entity-focused queries where a deep understanding of a specific entity and its relationships is required, as shown in the figure below.

![Local Search](./imgs/local-search.png)

When a user submits a query, the local search method first identifies semantically related entities within the knowledge graph using vector search. These entities act as entry points for retrieving relevant information, including directly connected entities, relationships, and summaries from community reports.

Text chunks from the input documents associated with these entities are also extracted. The retrieved data is ranked and filtered to fit within a constrained context window, ensuring that only the most relevant information is included in the final response.

To implement local search, we first need to calculate text embeddings for entities and create a vector index, which allows us to efficiently retrieve the most relevant entities based on the user query.

We will embed entity descriptions and relationships into a vector space so as to use similarity search to identify which entities are most closely related to the input.

Once these relevant entities are found, they serve as entry points to retrieve additional structured and unstructured data.

In [34]:
# Retrieve entities and their summaries
entities, _, _ = neo4j_driver.execute_query(
    """
MATCH (e:__Entity__)
RETURN e.summary AS summary, e.name AS name
"""
)

# Calculate embeddings based on entity summaries
data = [
    {
        'name': el['name'],
        'embedding': embed(el['summary'] if el['summary'] else "")[0]
    }
    for el in entities
]

# Store embeddings in the database
neo4j_driver.execute_query(
    """
UNWIND $data AS row
MATCH (e:__Entity__ {name: row.name})
CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)
""",
    data=data,
)

# Create vector index entities
neo4j_driver.execute_query(
    """
CREATE VECTOR INDEX entities IF NOT EXISTS
FOR (n:__Entity__)
ON (n.embedding)
""",
    data=data,
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002B211D6A390>, keys=[])

In [35]:
# There are quite few entities without summaries
print(f"Out of {len(entities)} entities, {sum(1 for el in entities if not el['summary'])} have no summary")

Out of 154 entities, 16 have no summary


Finally, the local search expands the initial set of relevant nodes, identified through vector search, to include their connected entities, text chunks, summaries, and relationships:

In [36]:
local_search_query = """
CALL db.index.vector.queryNodes('entities', $k, $embedding)
YIELD node, score
WITH collect(node) as nodes
WITH collect {
    UNWIND nodes as n
    MATCH (n)<-[:HAS_ENTITY]->(c:__Chunk__)
    WITH c, count(distinct n) as freq
    RETURN c.text AS chunkText
    ORDER BY freq DESC
    LIMIT $topChunks
} AS text_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[:IN_COMMUNITY]->(c:__Community__)
    WITH c, c.rank as rank, c.weight AS weight
    RETURN c.summary 
    ORDER BY rank, weight DESC
    LIMIT $topCommunities
} AS report_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m) 
    WHERE m IN nodes
    RETURN r.summary AS descriptionText
    ORDER BY r.rank, r.weight DESC 
    LIMIT $topInsideRels
} as insideRels,
collect {
    UNWIND nodes as n
    RETURN n.summary AS descriptionText
} as entities
RETURN {Chunks: text_mapping, Reports: report_mapping, 
       Relationships: insideRels, 
       Entities: entities} AS text
"""

All retrieved objects above, such as text chunks, community descriptions, relationships, and entity summaries, are ranked and limited to ensure the prompt remains
 manageable.
- Text chunks are ranked by how frequently they are associated with relevant entities and limited to the top `topChunks`. 
- Community descriptions are ordered by rank and weight, selecting only the `topCommunities`. 
- Relationships are ranked by their importance and limited to `topInsideRels`.
- Entity summaries are retrieved without additional ranking constraints to ensure only the most relevant information is included in the final response.


Lastly, we need to define the summarizing prompt for local search, which is the `LOCAL_SEARCH_SYSTEM_PROMPT` in `ch07_tools.py`. 

In [37]:
# Local search implementation
k_entities = 5

topChunks = 3
topCommunities = 3
topInsideRels = 3

def local_search(query: str) -> str:
    # Fetch context using the local search Cypher statement
    context, _, _ = neo4j_driver.execute_query(
        local_search_query,
        embedding=embed(query)[0],
        topChunks=topChunks,
        topCommunities=topCommunities,
        topInsideRels=topInsideRels,
        k=k_entities
    )

    # Stringify context for the prompt
    context_str = str(context[0]['text'])

    local_messages = [
        {
            'role': 'system',
            'content': ch07_tools.get_local_system_prompt(context_str)
        },
        {
            'role': 'user',
            'content': query
        }
    ]

    # Generate final response using local search context
    final_answer = chat(local_messages, model='gpt-5.1')

    return final_answer

In [38]:
print(local_search("Who is Ulysses?"))

## Identity and Role of Ulysses

Ulysses (the Latin name for the Greek Odysseus) is a mortal hero and king of Ithaca, renowned as a powerful and dangerous warrior who fought in the Trojan War and helped sack the city of Troy [Data: Entities (1, 4); Relationships (1)]. Before his disappearance, he was a leading figure among the men of Ithaca, commanding great respect, followers, and dependents, and acting as a central political and social authority in his homeland [Data: Entities (1)].

He is the husband of Penelope and the father of Telemachus, and the rightful master of the household in Ithaca. In his long absence after Troy, his house is overrun by suitors who consume his wealth and court his wife, even as they still fear the possibility of his return and revenge, remembering him as once formidable in war and leadership [Data: Entities (1)].

## Ulysses in the Trojan War

Ulysses was a key figure in the expedition against Troy. He sailed with the other Achaeans (Argives, Danaans) to 